### This is to test the lexical similarity, not semantic similarity.

In [ ]:
import os
from pathlib import Path
import joblib
import gc

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel


BASE_DIR = Path.cwd().parent

ABSTRACTS_PATH = BASE_DIR / "data" / "arxiv_metadata.csv"

VECTORIZER_PATH = BASE_DIR / "data" / "tfidf_vectorizer.joblib"
MATRIX_PATH = BASE_DIR / "data" / "tfidf_matrix.joblib"

In [ ]:
abstracts_df = pd.read_csv(ABSTRACTS_PATH, usecols=["id", "paper_id", "abstract"])
abstracts_df

/var/folders/kz/dlw3ql7s3ds1_72hwlw7_t540000gn/T/ipykernel_84101/1120635322.py:1: DtypeWarning: Columns (0: paper_id) have mixed types. Specify dtype option on import or set low_memory=False.
  abstracts_df = pd.read_csv(ABSTRACTS_PATH)


,id,paper_id,title,abstract,authors,categories,update_year
0,0,704.0009,"The Spitzer c2d Survey of Large, Nearby, Inste...",We discuss the results from the combined IRA...,"[""Harvey Paul"", ""Merin Bruno"", ""Huard Tracy L....","[""astro-ph""]",2010
1,1,704.0017,Spectroscopic Observations of the Intermediate...,Results from spectroscopic observations of t...,"[""Mhlahlo Nceba"", ""Buckley David H."", ""Dhillon...","[""astro-ph""]",2009
2,2,704.0023,ALMA as the ideal probe of the solar chromosphere,"The very nature of the solar chromosphere, i...","[""Loukitcheva M. A."", ""Solanki S. K."", ""White ...","[""astro-ph""]",2009
3,3,704.0044,Astrophysical gyrokinetics: kinetic and fluid ...,We present a theoretical framework for plasm...,"[""Schekochihin A. A."", ""Cowley S. C."", ""Dorlan...","[""astro-ph"", ""nlin.CD"", ""physics.plasm-ph"", ""p...",2015
4,4,704.0048,Inference on white dwarf binary systems using ...,We report on the analysis of selected single...,"[""Stroeer Alexander"", ""Veitch John"", ""Roever C...","[""gr-qc"", ""astro-ph""]",2008
...,...,...,...,...,...,...,...
381339,381339,quant-ph/9903043,A Possible Anisotropy in Blackbody Radiation V...,A non-local gauge symmetry of a complex scal...,"[""Dastidar T K Rai""]","[""quant-ph"", ""astro-ph"", ""hep-th""]",2007
381340,381340,quant-ph/9903053,Father Time. I. Does the Cosmic Microwave Back...,The existence of a non-thermodynamic arrow o...,"[""Dastidar T K Rai""]","[""quant-ph"", ""astro-ph"", ""hep-th""]",2009
381341,381341,quant-ph/9907088,On Bures fidelity of displaced squeezed therma...,Fidelity plays a key role in quantum informa...,"[""Wang Xiang-Bin"", ""Oh C. H."", ""Kwek L. C.""]","[""quant-ph"", ""astro-ph""]",2008
381342,381342,solv-int/9404002,Dynamical Systems Accepting the Normal Shift,Newtonian dynamical systems accepting the no...,"[""Sharipov R. A.""]","[""solv-int"", ""alg-geom"", ""astro-ph"", ""gr-qc"", ...",2008


In [ ]:
def build_tfidf_matrix(df: pd.DataFrame):

    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents='unicode', # swtich to 'ascii' for faster processing
        analyzer='word',
        stop_words=None, # see https://scikit-learn.org/stable/modules/feature_extraction.html#stop-words
        max_df=0.95, # maximum document frequency (ignore terms that appear in more than 95% of documents)
        min_df=1, # minimum document frequency (ignore terms that appear in less than 1 documents)
        max_features=10000,
        norm='l2',
        ngram_range=(1,2), # unigrams and bigrams
        )

    tfidf_matrix = vectorizer.fit_transform(df['abstract'])
    return vectorizer, tfidf_matrix

vectorizer, tfidf_matrix = build_tfidf_matrix(abstracts_df) # build the TF-IDF matrix

# del abstracts_df # save memory
# gc.collect()

1013

In [11]:
joblib.dump(vectorizer, VECTORIZER_PATH)
joblib.dump(tfidf_matrix, MATRIX_PATH)

['/Users/salirafi/Documents/Personal Project/Abstract Recommender/data/tfidf_matrix.joblib']

Test the result.

In [ ]:
def recommend_papers_tfidf(query_text: str, metadata: pd.DataFrame, top_k: int = 10) -> pd.DataFrame:

    query_text = str(query_text).strip()
    if not query_text:
        raise ValueError("Query text is empty.")

    query_vec = vectorizer.transform([query_text])

    # cosine similarity because vectors are L2-normalized
    scores = linear_kernel(query_vec, tfidf_matrix).flatten()

    top_indices = np.argsort(scores)[::-1][:top_k] # get indices of top-k scores
    results = metadata.iloc[top_indices].copy()
    results["score"] = scores[top_indices]

    return results[
        ["score", "title", "categories", "abstract", "update_year"]
    ].sort_values(by="update_year", ascending=False).reset_index(drop=True)

In [21]:
query_text = """
Transmission spectroscopy of exoplanets using ground-based high resolution spectrographs.
"""
res = recommend_papers_tfidf(query_text, abstracts_df)

In [ ]:
res

# looks promising

,score,title,categories,abstract,update_year
0,0.407944,A multi-object approach for studying exoplanet...,"[""astro-ph.EP"", ""astro-ph.IM""]",Atmospheric characterization of exoplanets has...,2025
1,0.329217,High-Resolution Echelle Spectroscopy for Solar...,"[""astro-ph.EP"", ""astro-ph.IM""]",Transmission spectroscopy has proven to be an ...,2025
2,0.315925,Open Source High-Resolution Exoplanet Atmosphe...,"[""astro-ph.EP"", ""astro-ph.IM""]","High-resolution spectroscopy (R > 25,000) has ...",2025
3,0.323364,Robustness Measures for Molecular Detections u...,"[""astro-ph.EP""]",Ground-based high-resolution transmission sp...,2023
4,0.332232,Broadband transmission spectroscopy of HD20945...,"[""astro-ph.EP""]",The detection and characterization of exopla...,2021
5,0.350910,Exoplanets Sciences with Nulling Interferomete...,"[""astro-ph.IM"", ""astro-ph.EP""]",Understanding the atmospheres of exoplanets ...,2020
6,0.422219,LRG-BEASTS III: Ground-based transmission spec...,"[""astro-ph.EP""]",We have performed ground-based transmission ...,2017
7,0.498399,Regaining the FORS: making optical ground-base...,"[""astro-ph.IM"", ""astro-ph.EP"", ""astro-ph.SR"", ...",Transmission spectroscopy facilitates the de...,2016
8,0.347726,Regaining the FORS: optical ground-based trans...,"[""astro-ph.EP"", ""astro-ph.IM""]","In the past few years, the study of exoplane...",2015
9,0.313646,High Resolution Transmission Spectroscopy as a...,"[""astro-ph.EP""]",We present high resolution transmission spec...,2015
